In [10]:
import pandas as pd
import sqlite3
import os

def setup_firm_database(
    firm_csv_path='../green cleaned.csv', 
    finbert_csv_path='../examples/monthly_signals_decay.csv', 
    db_path='firm_data.db'
):
    print("1. Loading firm quantitative data...")
    # Load firm data
    df_firm = pd.read_csv(firm_csv_path, dtype={'ncusip': 'string'})
    
    # Calculate ret_fwd_1 (must be done on the full dataset, grouped by permno)
    if 'permno' in df_firm.columns and 'ret_excess' in df_firm.columns:
        df_firm = df_firm.sort_values(by=['permno', 'datadate'])
        df_firm['ret_fwd_1'] = df_firm.groupby('permno')['ret_excess'].shift(-1)
    else:
        print("Warning: 'permno' or 'ret_excess' not found. Cannot calculate ret_fwd_1.")

    # Create a year_month column to match the FinBERT dataset format (YYYY-MM)
    df_firm['datadate'] = pd.to_datetime(df_firm['datadate'])
    df_firm['year_month'] = df_firm['datadate'].dt.strftime('%Y-%m')
    
    print("2. Loading FinBERT sentiment data...")
    # Load FinBERT data
    df_finbert = pd.read_csv(finbert_csv_path)
    
    # Rename columns to match the agent's expected environment and the merge keys
    # Mapping 'symbol' to 'ticker', 'weighted_sentiment_score' to 'finbert_score', 'signal' to 'finbert_sentiment'
    df_finbert = df_finbert.rename(columns={
        'symbol': 'ticker',
        'weighted_sentiment_score': 'finbert_score',
        'signal': 'finbert_sentiment'
    })
    
    # Select only the needed FinBERT columns to prevent database bloat
    finbert_cols_to_keep = ['ticker', 'year_month', 'finbert_score', 'finbert_sentiment']
    df_finbert = df_finbert[finbert_cols_to_keep]

    print("3. Merging datasets...")
    # Merge the fundamental data with the FinBERT data using left join
    merged_df = pd.merge(df_firm, df_finbert, on=['ticker', 'year_month'], how='left')
    
    # Fill missing sentiment with neutral values for firms without news coverage that month
    merged_df['finbert_score'] = merged_df['finbert_score'].fillna(0.0)
    merged_df['finbert_sentiment'] = merged_df['finbert_sentiment'].fillna('hold')

    # Revert datadate to string format for SQLite storage consistency
    merged_df['datadate'] = merged_df['datadate'].dt.strftime('%Y-%m-%d')
    
    # Filter final columns for the database
    final_columns = [
        'datadate', 'ticker', 'permno', 'bm', 'mom12m', 'mve', 
        'ret_fwd_1', 'finbert_score', 'finbert_sentiment'
    ]
    
    # Ensure only available columns are kept (in case some are missing from the raw green CSV)
    final_columns = [col for col in final_columns if col in merged_df.columns]
    df_final = merged_df[final_columns].copy()

    print("4. Writing to SQLite database...")
    if os.path.exists(db_path):
        os.remove(db_path)
        
    conn = sqlite3.connect(db_path)
    
    # Write to SQLite in chunks to manage memory efficiently
    chunk_size = 50000
    for i in range(0, len(df_final), chunk_size):
        chunk = df_final.iloc[i:i + chunk_size]
        if i == 0:
            chunk.to_sql('firms', conn, if_exists='replace', index=False)
        else:
            chunk.to_sql('firms', conn, if_exists='append', index=False)
            
    cursor = conn.cursor()
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_ticker ON firms(ticker)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_date ON firms(datadate)")
    conn.commit()
    conn.close()
    
    print(f"✓ Database created: {db_path} ({len(df_final):,} rows)")
    print(f"✓ Columns available to Agents: {final_columns}")
    
    return final_columns

In [11]:
setup_firm_database('../green cleaned.csv')

1. Loading firm quantitative data...
2. Loading FinBERT sentiment data...
3. Merging datasets...
4. Writing to SQLite database...
✓ Database created: firm_data.db (269,255 rows)
✓ Columns available to Agents: ['datadate', 'ticker', 'permno', 'bm', 'mom12m', 'mve', 'ret_fwd_1', 'finbert_score', 'finbert_sentiment']


['datadate',
 'ticker',
 'permno',
 'bm',
 'mom12m',
 'mve',
 'ret_fwd_1',
 'finbert_score',
 'finbert_sentiment']

In [13]:
import sqlite3
import pandas as pd
from crewai.tools import tool
from crewai import Agent, Task, Crew, Process
from crewai import LLM
import time
from functools import wraps
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from datetime import datetime

# ============================================
# Global Walk-Forward State
# ============================================
PIPELINE_STATE = {
    "current_eval_date": None,  
    "allow_returns": True       
}

# ============================================
# Rate Limiting & Tools Setup
# ============================================
class RateLimiter:
    def __init__(self, calls_per_minute=5):
        self.calls_per_minute = calls_per_minute
        self.interval = 60.0 / calls_per_minute
        self.last_call = 0
        self.call_count = 0
        
    def wait_if_needed(self):
        current_time = time.time()
        time_since_last = current_time - self.last_call
        if time_since_last < self.interval:
            sleep_time = self.interval - time_since_last
            time.sleep(sleep_time)
        self.last_call = time.time()
        self.call_count += 1

rate_limiter = RateLimiter(calls_per_minute=5)

def rate_limit(func):
    @wraps(func)
    @retry(
        retry=retry_if_exception_type((Exception,)),
        wait=wait_exponential(multiplier=1, min=4, max=60),
        stop=stop_after_attempt(5)
    )
    def wrapper(*args, **kwargs):
        rate_limiter.wait_if_needed()
        try:
            return func(*args, **kwargs)
        except Exception as e:
            raise
    return wrapper

@tool("query_firm_database")
@rate_limit
def query_firm_database(sql_query: str) -> str:
    """Execute SQL queries on the firms database."""
    query_lower = sql_query.lower()
    if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in query_lower:
        return f"SECURITY BLOCK: Access to 'ret_fwd_1' is strictly prohibited during evaluation for {PIPELINE_STATE['current_eval_date']}."
    
    try:
        conn = sqlite3.connect('firm_data.db')
        df = pd.read_sql_query(sql_query, conn)
        conn.close()
        
        if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in df.columns:
            df = df.drop(columns=["ret_fwd_1"])
            
        if df.empty: return "Query returned no results."
        result = f"Query returned {len(df)} rows:\n\n" + df.head(5).to_string(index=False)
        return result
    except Exception as e:
        return f"Error executing query: {str(e)}"

@tool("get_database_schema")
@rate_limit
def get_database_schema() -> str:
    """Get the schema of the firms database."""
    try:
        conn = sqlite3.connect('firm_data.db')
        cursor = conn.cursor()
        cursor.execute("PRAGMA table_info(firms)")
        columns = cursor.fetchall()
        conn.close()
        
        schema = "Database: firm_data.db\nTable: firms\nColumns:\n"
        for col in columns:
            if not PIPELINE_STATE["allow_returns"] and col[1] == "ret_fwd_1":
                continue 
            schema += f"  - {col[1]} ({col[2]})\n"
        return schema
    except Exception as e:
        return f"Error retrieving schema: {str(e)}"

@tool("test_complex_condition")
@rate_limit
def test_complex_condition(sql_condition: str, date: str = None) -> str:
    """
    Test complex conditions with AND/OR/NOT logic.
    CRITICAL: Pass ONLY the condition (e.g., 'bm > 1.0 AND mom12m < 0.5'). 
    DO NOT pass a full SELECT statement.
    For 'date', you can pass a single date ('YYYY-MM-DD') or a range ('YYYY-MM-DD to YYYY-MM-DD').
    """
    # Auto-correct if the LLM accidentally passes a full SQL query
    if "WHERE " in sql_condition.upper():
        sql_condition = sql_condition[sql_condition.upper().find("WHERE ") + 6:]
    elif "SELECT " in sql_condition.upper():
        return "Error: Please pass ONLY the condition string (e.g., 'mve > 1.0'). Do not pass a SELECT query."

    if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in sql_condition.lower():
        return f"SECURITY BLOCK: You cannot use 'ret_fwd_1' during zero-shot evaluation for {PIPELINE_STATE['current_eval_date']}."
        
    try:
        conn = sqlite3.connect('firm_data.db')
        target_date = date if date else PIPELINE_STATE["current_eval_date"]
        
        # Parse date range vs single date
        if " to " in target_date:
            start_date, end_date = target_date.split(" to ")
            date_clause = f"datadate BETWEEN '{start_date.strip()}' AND '{end_date.strip()}'"
        else:
            date_clause = f"datadate = '{target_date.strip()}'"
        
        count_query = f"SELECT COUNT(*) as n_firms FROM firms WHERE {date_clause} AND ({sql_condition})"
        columns_to_select = "datadate, ticker, mve, bm, mom12m, finbert_score, finbert_sentiment" if 'ticker' in sql_condition else "*"
        sample_query = f"SELECT {columns_to_select} FROM firms WHERE {date_clause} AND ({sql_condition}) LIMIT 5"
        
        n_firms = pd.read_sql_query(count_query, conn)['n_firms'].values[0]
        sample_df = pd.read_sql_query(sample_query, conn)
        conn.close()
        
        if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in sample_df.columns:
            sample_df = sample_df.drop(columns=["ret_fwd_1"])
            
        result = f"Condition Test at {target_date}:\nFirms meeting condition: {n_firms:,}\n\n"
        result += sample_df.to_string(index=False) if not sample_df.empty else "(No firms meet this condition)"
        return result
    except Exception as e:
        return f"Error testing condition: {str(e)}"

def get_rate_limited_llm():
    """Get Local Ollama LLM using CrewAI's native LiteLLM wrapper."""
    llm = LLM(
        model="ollama/llama3.1",
        base_url="http://localhost:11434", # Default Ollama port
        timeout=720 # Give the local model extra time to think
    )
    return llm

# ============================================
# ORIGINAL INSTRUCTIONS 
# ============================================
QUANT_BACKSTORY = """
You are an expert quantitative strategist who creates systematic, rule-based trading strategies.
    
CRITICAL DATA UNDERSTANDING:
- 'mve' = log(market value of equity), represents log firm size. Standardized to have mean 0, variance 1.
- 'bm' = book-to-market ratio (value factor). High = undervalued, Low = overvalued. Standardized to have mean 0, variance 1.
- 'mom12m' = 12-month momentum. Standardized to have mean 0, variance 1.
- 'finbert_score' = Time-decay weighted SUM of sentiment scores from news articles. Can exceed +/- 1.0. A score of exactly 0.0 usually means no news coverage.
- 'finbert_sentiment' = Categorical label ('buy', 'sell', 'hold') based on the finbert_score.

PORTFOLIO SIZE CONSTRAINT:
Do not use excessively large z-scores (like > 3 or < -3) combined with AND statements. 

Your task is to develop EXPLICIT, SYSTEMATIC RULES for generating trading signals. 
    
1. EXPLORE THE DATA:
- Identify extreme values for the quantitative factors.
- Look at the distribution of finbert_score.
    
2. DEVELOP CLEAR RULES based on economic intuition:
- Value Traps: Should high bm (cheap) stocks be avoided if finbert_score is heavily negative?
- Momentum Confirmation: Does high mom12m perform better when backed by a positive finbert_score?
- No-News Firms: How will you handle firms with finbert_score = 0.0?
    
3. DEFINE SPECIFIC THRESHOLDS:
 Your output must include exact rules. You can use test_complex_condition to test combinations.
 Examples of format (DO NOT COPY THESE EXACT VALUES):
- "BUY if: bm > 0.71 AND finbert_score > 0.15" 
- "BUY if: (mom12m > 0.85 AND finbert_score > 0.0) OR (bm > 1.2 AND finbert_sentiment = 'buy')"
- "SELL if: bm < -1.25 OR finbert_score < -0.5" 
- "HOLD: all other cases"
- Be PRECISE in your thresholds.
    
4. PROVIDE RATIONALE:
- Why these thresholds? What's the economic intuition for combining the factor with the sentiment?
    
CRITICAL REQUIREMENTS:
- Rules must be DETERMINISTIC (same inputs → same output)
- Use strict SQL-compatible syntax for conditions (>, <, AND, OR, =)
- When querying finbert_sentiment, use single quotes (e.g., finbert_sentiment = 'buy')
"""

QUANT_EXPECTED_OUTPUT = """
Complete strategy document with:
1. Data exploration summary
2. Explicit BUY rule with thresholds
3. Explicit SELL rule with thresholds  
4. HOLD rule (default case)
5. Economic rationale for each rule

OUTPUT FORMAT:
===========================================
SYSTEMATIC TRADING RULES
===========================================

Data Exploration Summary:
- [Key statistics and patterns observed]

BUY RULE:
if [ANY complex condition using AND/OR/NOT]:
    signal = BUY

Examples of valid BUY rules:
- "bm > 1.15 AND finbert_score > 0.2" (Value with positive news)
- "mom12m > 0.8 OR finbert_sentiment = 'buy'" (Momentum OR strong news)
- "(bm > 1.02 AND mom12m > 0.53) AND finbert_score != 0.0" (Multi-factor requiring news coverage)
- "mve < -0.5 AND finbert_score > 1.5" (Small cap with high news volume/sentiment)

SELL RULE:
if [ANY complex condition]:
    signal = SELL

HOLD RULE:
else:
    signal = HOLD

Rationale:
- [Economic reasoning for BUY rule]
- [Economic reasoning for SELL rule]

===========================================
"""

# ============================================
# ACE Agents (max_iter reduced to 5)
# ============================================
generator_agent = Agent(
    role="Quantitative Strategy Developer (Generator)",
    goal="Develop systematic trading rules using the current rulebook, test them against historical forward returns ('ret_fwd_1'), and record reasoning.",
    backstory=QUANT_BACKSTORY + """
    During this training phase, apply the active ACE rulebook to generate BUY/HOLD/SELL signals on historical data. 
    You MUST query 'ret_fwd_1' to evaluate if your signals accurately predicted the forward returns. 
    Record your step-by-step reasoning and explicitly flag which rules from the playbook proved useful and which were misleading when compared against known outcomes.
    """,
    tools=[get_database_schema, query_firm_database, test_complex_condition],
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5, # Reduced from 15
    max_rpm=10
)

sentiment_agent = Agent(
    role="Narrative & Sentiment Specialist",
    goal="Analyze how FinBERT textual sentiment interacts with quantitative factor anomalies, and propose sentiment filters.",
    backstory="""You are an expert in financial NLP. You review candidate quantitative rules and check them against 'finbert_score' and 'finbert_sentiment'. 
    Your job is to:
    1. Filter 'Value Traps' (high bm but highly negative finbert_score).
    2. Confirm 'Momentum Quality' (high mom12m supported by positive finbert_score).
    3. Suggest joint thresholds (e.g., AND finbert_score > 0.5) to improve forward returns.
    Remember that a finbert_score of 0.0 usually means no news, so decide if you want to include or exclude no-news firms.""",
    tools=[get_database_schema, query_firm_database, test_complex_condition],
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5,
    max_rpm=10
)

reflector_agent = Agent(
    role="Strategy Reflector",
    goal="Analyze the generator's record, identify error patterns, and propose targeted delta updates.",
    backstory="You are an expert performance analyst. Review the Generator's reasoning and the actual realized outcomes ('ret_fwd_1') to diagnose prediction failures. Propose small, targeted 'delta' additions or modifications to the rulebook based on identified patterns.",
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5, # Reduced from 15
    max_rpm=10
)

curator_agent = Agent(
    role="Rulebook Curator",
    goal="Integrate delta updates into the master rulebook while preventing context collapse.",
    backstory="You manage the master rulebook. Integrate proposed deltas from the Reflector by appending new rules, updating existing ones based on performance, and removing duplicates. Modify only small portions of the context to prevent context collapse.",
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5, # Reduced from 15
    max_rpm=10
)

evaluation_agent = Agent(
    role="Quantitative Strategy Developer (Zero-Shot Evaluator)",
    goal="Execute the optimized rulebook on out-of-sample data to generate final trading signals for all S&P 500 firms.",
    backstory=QUANT_BACKSTORY + """
    You must use causal masking to prevent any look-ahead bias. You are in the final evaluation phase. You must take the finalized ACE strategy rulebook and apply it strictly to the current evaluation date. You DO NOT have access to future returns. 
    Output the final explicit thresholds so they can be implemented in pandas.
    """,
    tools=[get_database_schema, query_firm_database, test_complex_condition],
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5, # Reduced from 15
    max_rpm=10
)

# ============================================
# Core Functions
# ============================================
def run_ace_loop(training_start, training_end, iterations=3):
    current_rulebook = """BASE RULEBOOK v1.0: 
- Identify z-score combinations for value (bm), momentum (mom12m), and size (mve).
- Incorporate textual sentiment ('finbert_score' and 'finbert_sentiment') to filter value traps and confirm momentum.
- Remember 'finbert_score' is a sum and can exceed +/- 1.0. A score of exactly 0.0 usually means no news.
- Avoid extreme conditions that yield empty portfolios.
"""
    
    for i in range(iterations):
        print(f"\n--- ACE Iteration {i+1} ---")
        
        generate_task = Task(
            description=f"""Apply this rulebook to the training data ({training_start} to {training_end}).
            Test conditions using test_complex_condition, query 'ret_fwd_1' to check your hypotheses, and record reasoning.
            CURRENT RULEBOOK:
            {current_rulebook}""",
            expected_output="A detailed log of predictions, the actual 'ret_fwd_1' outcomes, your economic rationale, and flagged rules (useful vs misleading).",
            agent=generator_agent
        )

        sentiment_task = Task(
            description="""Review the Generator's proposed rules. Use test_complex_condition to test adding 'finbert_score' or 'finbert_sentiment' conditions to them. 
            Evaluate if combining sentiment with the quant factors improves the 'ret_fwd_1' outcomes on the training data. Output proposed modifications.""",
            expected_output="Diagnostic evaluation of factor-sentiment interaction and proposed joint rules.",
            agent=sentiment_agent
        )
        
        reflect_task = Task(
            description="""Analyze the Generator's execution log. 
            CRITICAL: If the Generator proposed a rule that returned 0 firms, aggressively flag it as a FAILURE due to extreme thresholds. 
            Output targeted 'delta' updates to lower the thresholds so more firms are captured, while maintaining the economic edge.""",
            expected_output="A structured list of specific delta updates.",
            agent=reflector_agent
        )
        
        curate_task = Task(
            description=f"Integrate the Reflector's delta updates into the current rulebook.\nCURRENT RULEBOOK:\n{current_rulebook}\nOutput the complete updated master rulebook.",
            expected_output="The updated, structured master rulebook with explicit thresholds and economic rationale.",
            agent=curator_agent
        )
        
        ace_crew = Crew(
            agents=[generator_agent, sentiment_agent, reflector_agent, curator_agent],
            tasks=[generate_task, sentiment_task, reflect_task, curate_task],
            process=Process.sequential,
            verbose=False
        )
        current_rulebook = ace_crew.kickoff()
        
    return current_rulebook

def generate_zero_shot_signals(rulebook, eval_date):
    eval_task = Task(
        description=f"""Apply this optimized rulebook strictly to the datadate = '{eval_date}'.
        1. Explore extreme values for each characteristic at this date.
        2. Use test_complex_condition to verify your rule combinations.
        3. Develop systematic rules with specific z-score thresholds.
        
        OPTIMIZED RULEBOOK:
        {rulebook}
        """,
        expected_output=QUANT_EXPECTED_OUTPUT,
        agent=evaluation_agent
    )
    
    eval_crew = Crew(
        agents=[evaluation_agent],
        tasks=[eval_task],
        process=Process.sequential,
        verbose=True
    )
    return eval_crew.kickoff()


# ============================================
# Main Execution
# ============================================
if __name__ == "__main__":
    
    # 1. Setup Database
    # variables = ['datadate', 'ticker', 'bm', 'mom12m', 'mve', 'ret_excess']
    # setup_firm_database('../green cleaned.csv', selected_columns=variables)
    
    # Define your monthly runs. 
    # Adjust the train_start and train_end to reflect your desired rolling window (e.g., trailing 12 months)
    evaluation_runs = [
        #{"eval_date": "2024-01-31", "train_start": "2023-01-01", "train_end": "2023-12-31"},
         #{"eval_date": "2024-02-29", "train_start": "2023-02-01", "train_end": "2024-01-31"},
         {"eval_date": "2024-03-31", "train_start": "2023-03-01", "train_end": "2024-02-29"},
    ]
    
    for run in evaluation_runs:
        eval_date_str = run["eval_date"]
        print(f"\n{'='*70}\n EXECUTING PIPELINE FOR {eval_date_str}\n{'='*70}")
        
        PIPELINE_STATE["current_eval_date"] = eval_date_str
        training_start = run["train_start"]
        training_end = run["train_end"]
        
        # PHASE 1: TRAINING (ACE Loop)
        PIPELINE_STATE["allow_returns"] = True
        print(f"Training ACE Rulebook from {training_start} to {training_end}...")
        optimized_rulebook = run_ace_loop(training_start, training_end, iterations=3)
        
        # PHASE 2: ZERO-SHOT EVAL
        PIPELINE_STATE["allow_returns"] = False
        print(f"\nApplying finalized rulebook zero-shot to {PIPELINE_STATE['current_eval_date']}...")
        final_strategy = generate_zero_shot_signals(optimized_rulebook, PIPELINE_STATE['current_eval_date'])
        
        # Calculate the next month and year for the filename
        eval_dt = datetime.strptime(eval_date_str, "%Y-%m-%d")
        if eval_dt.month == 12:
            next_month = 1
            next_year = eval_dt.year + 1
        else:
            next_month = eval_dt.month + 1
            next_year = eval_dt.year
            
        # Format filename as systematic_strategy_MM_YYYY_llama.txt
        filename = f'systematic_strategy_{next_month:02d}_{next_year}_llama.txt'
        
        with open(filename, 'w') as f:
            f.write(f"--- OPTIMIZED ACE RULEBOOK ---\n{optimized_rulebook}\n\n")
            f.write(f"--- FINAL ZERO-SHOT STRATEGY ---\n{final_strategy}")
            
        print(f"✓ Strategy saved to {filename}")


 EXECUTING PIPELINE FOR 2024-03-31
Training ACE Rulebook from 2023-03-01 to 2024-02-29...

--- ACE Iteration 1 ---


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantitative Strategy Developer (Generator)                                                             │
│                                                                                                                 │
│  Task: Apply this rulebook to the training data (2023-03-01 to 2024-02-29).                                     │
│              Test conditions using test_complex_condition, query 'ret_fwd_1' to check your hypotheses, and      │
│  record reasoning.                                                                                              │
│              CURRENT RULEBOOK:                                                                                  │
│              BASE RULEBOOK v1.0:                                                                                │
│  - Identify z-score combinations for value (bm), momentum (mom12m), and size (mve).                             │
│  - Incorporate textual sentiment ('finbert_score' and 'finbert_sentiment') to filter value traps and confirm    │
│  momentum.                                                                                                      │
│  - Remember 'finbert_score' is a sum and can exceed +/- 1.0. A score of exactly 0.0 usually means no news.      │
│  - Avoid extreme conditions that yield empty portfolios.                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

Exception in thread Thread-25 (get_input):
Traceback (most recent call last):
  File "/Users/nathansun/opt/anaconda3/envs/agentics/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/Users/nathansun/opt/anaconda3/envs/agentics/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 788, in run_closure
    _threading_Thread_run(self)
  File "/Users/nathansun/opt/anaconda3/envs/agentics/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/nathansun/opt/anaconda3/envs/agentics/lib/python3.12/site-packages/crewai/events/listeners/tracing/utils.py", line 359, in get_input
    response = input().strip().lower()
               ^^^^^^^
  File "/Users/nathansun/opt/anaconda3/envs/agentics/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 1475, in raw_input
    self._shell_parent_ident.get(),
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
LookupError: <ContextVar name='shell_parent_ident' at 0x1122d01d0>


APIConnectionError: litellm.APIConnectionError: OllamaException - [Errno 61] Connection refused